# Prototype Design Pattern (Classic OOP)

In Java, you typically define a `Prototype` interface with a `clone()` method and implement it in concrete classes. We will replicate that structure here, avoiding Python shortcuts like dataclasses or purely functional approaches.

### THE INTERFACE (Java's "implements Prototype")

In [4]:
from abc import ABC, abstractmethod

class Prototype(ABC):
    """
    The Prototype interface/
    declares the clone method.
    In Java, this would be: public interface Prototype { Prototype clone(); }
    """
    @abstractmethod
    def clone(self):
        pass

### CONCRETE PROTOTYPE

In [5]:
import copy
class Shape(Prototype):
    """
    A Concrete Class.
    """
    def __init__(self, x, y, color):
        # Standard explicit constructor (like Java)
        self.x = x
        self.y = y
        self.color = color

    def clone(self):
        """
        Java-style implementation:
        1. Create a copy (using deepcopy to ensure references are safe).
        2. Cast/Return it as the Interface type.
        """
        return copy.deepcopy(self)

    def __str__(self):
        return f"Shape [x={self.x}, y={self.y}, color={self.color}]"

### CLIENT CODE

In [6]:
def main():
    # 1. Create the original object (The "Registry" item)
    print("--- Creating Original ---")
    circle = Shape(10, 20, "Red")
    print("Original: " + str(circle))

    # 2. Clone it (Java style: calling the .clone() method)
    print("\n--- Cloning Object ---")
    another_circle = circle.clone()
    
    # 3. Modify the clone
    another_circle.x = 99
    another_circle.color = "Blue"

    # 4. Verification
    print("Original: " + str(circle))       # Should remain Red/10
    print("Clone:    " + str(another_circle)) # Should be Blue/99

    # Identity Check
    print(f"\nAre they the same object? {circle is another_circle}") # False

if __name__ == "__main__":
    main()

--- Creating Original ---
Original: Shape [x=10, y=20, color=Red]

--- Cloning Object ---
Original: Shape [x=10, y=20, color=Red]
Clone:    Shape [x=99, y=20, color=Blue]

Are they the same object? False


#### Why this feels like Java

- Explicit Interface: We use ABC to strictly enforce the clone method, similar to implementing an interface in Java.
- Explicit Constructor: We use __init__ with explicit assignments (self.x = x) instead of Python's dataclass shortcut.
- Strict Method Definition: The clone method is explicitly defined in the class, wrapping the internal copy logic.

# Prototype Design Pattern (Pythonic Way)

#### The Concept
The Prototype pattern is used when creating a new object from scratch is expensive (e.g., database queries, complex calculations). Instead of building a new object, you `clone` (copy) an existing one and modify it slightly.

Think of it like a `"Save As..."` feature in a document editor. You don't type the whole document again; you copy the old one and change the title.

The Code (Pythonic Way)
In Python, we don't need a complex `Cloneable` interface like in Java. We simply use the built-in `copy` module.

### THE PROTOTYPE CLASS

In [7]:
import copy
from dataclasses import dataclass, field

@dataclass
class Robot:
    name: str
    model: str
    skills: List[str] = field(default_factory=list)
    battery_level: int = 100

    def clone(self) -> "Robot":
        """
        Creates a deep copy of the robot.
        We use deepcopy to ensure that modifying the 'skills' list
        of the clone does NOT affect the original robot.
        """
        return copy.deepcopy(self)

    def __str__(self):
        return (f"🤖 [{self.name}] Model: {self.model} | "
                f"Battery: {self.battery_level}% | Skills: {self.skills}")

### CLIENT CODE

In [8]:
def main():
    # 1. Create a "Base" Prototype (Expensive operation simulation)
    # Imagine this step involves downloading AI models or heavy config
    print("--- Creating Prototype ---")
    base_robot = Robot(name="Prototype-01", model="T-800")
    base_robot.skills.append("Walk")
    base_robot.skills.append("Talk")
    
    print(f"Original: {base_robot}\n")

    # 2. Clone the Prototype to create a new Robot
    # This is much faster than setting up a new robot from scratch
    print("--- Cloning Robot ---")
    worker_robot = base_robot.clone()
    
    # 3. Customize the Clone
    worker_robot.name = "Worker-Bot-A"
    worker_robot.skills.append("Weld") # Add a new skill only for this clone
    
    # 4. Create another Clone (Fighter)
    fighter_robot = base_robot.clone()
    fighter_robot.name = "Fighter-Bot-X"
    fighter_robot.skills.append("Fight")
    fighter_robot.model = "T-1000"

    # ==========================================
    # 3. VERIFICATION
    # ==========================================
    print(f"Clone 1:  {worker_robot}")
    print(f"Clone 2:  {fighter_robot}")
    
    # Verify the Original is untouched (Deep Copy worked)
    print(f"\nOriginal Integrity Check: {base_robot}")
    # If we didn't use deepcopy, 'Weld' and 'Fight' would have appeared here!

if __name__ == "__main__":
    main()

--- Creating Prototype ---
Original: 🤖 [Prototype-01] Model: T-800 | Battery: 100% | Skills: ['Walk', 'Talk']

--- Cloning Robot ---
Clone 1:  🤖 [Worker-Bot-A] Model: T-800 | Battery: 100% | Skills: ['Walk', 'Talk', 'Weld']
Clone 2:  🤖 [Fighter-Bot-X] Model: T-1000 | Battery: 100% | Skills: ['Walk', 'Talk', 'Fight']

Original Integrity Check: 🤖 [Prototype-01] Model: T-800 | Battery: 100% | Skills: ['Walk', 'Talk']


#### Why copy.deepcopy()?

If your object contains lists or other objects (like skills: List[str]), a normal copy (shallow copy) only copies the reference to the list.
- Shallow Copy: If you add "Weld" to the clone's skill list, the original robot also learns to weld. (Bad!)
- Deep Copy: The clone gets a completely new list. Changes don't affect the original.

#### When to use this?

- Performance: When __init__ is very slow (e.g., it connects to a database or parses a huge XML file).
- Configuration: When you have a complex default configuration (like a "Standard User Permission Set") and you want to create a new user by just tweaking the standard set.
